# 🌡️ Daily Climate Time Series Forecasting using Deep Learning
## ANN, RNN, LSTM, GRU - Regression Task
**Dataset:** Daily Delhi Climate (Train + Test)

This notebook forecasts mean temperature using ANN, RNN, LSTM, and GRU models.

---
## 1. 📦 Import Libraries

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f"TensorFlow: {tf.__version__}")


---
## 2. 📊 Dataset Understanding

In [ ]:

train_df = pd.read_csv('Daily Climate Time Series Data/DailyDelhiClimateTrain.csv')
test_df = pd.read_csv('Daily Climate Time Series Data/DailyDelhiClimateTest.csv')

print("=" * 60)
print("TRAINING DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {train_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nData Types:\n{train_df.dtypes}")
print(f"\nBasic Statistics:\n{train_df.describe()}")
print(f"\nMissing Values:\n{train_df.isnull().sum()}")
print(f"\nTest Set Shape: {test_df.shape}")

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric features: {numeric_cols}")
print(f"Target variable: meantemp (continuous - regression task)")

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(train_df.isnull(), cbar=True, yticklabels=False, ax=axes[0], cmap='viridis')
axes[0].set_title('Missing Value Heatmap', fontweight='bold')

train_df['meantemp'].hist(bins=30, ax=axes[1], color='#FF6B6B', edgecolor='black')
axes[1].set_title('Mean Temperature Distribution', fontweight='bold')
axes[1].set_xlabel('Temperature')
plt.tight_layout()
plt.savefig('climate_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(train_df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig('climate_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Time series plots
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for i, col in enumerate(numeric_cols):
    ax = axes[i // 2, i % 2]
    ax.plot(train_df['date'], train_df[col], alpha=0.8, linewidth=0.8, label='Train')
    ax.plot(test_df['date'], test_df[col], alpha=0.8, linewidth=0.8, label='Test', color='red')
    ax.set_title(f'{col} Over Time', fontweight='bold')
    ax.legend()
plt.suptitle('Climate Features Over Time', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('climate_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 3. 🔧 Preprocessing

**Steps explained:**
- **Combine train/test** for consistent scaling
- **MinMaxScaler (0-1):** Normalizes features - essential for RNN convergence
- **Sequence windowing:** Create sliding windows of past N timesteps to predict next value
  This transforms the problem into a supervised learning task
- **Handle missing values** with forward fill

In [ ]:

# Combine data
df = pd.concat([train_df, test_df], ignore_index=True)
df = df.sort_values('date').reset_index(drop=True)

# Handle missing/anomalous values
df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(method='bfill')

# Fix meanpressure outliers (values < 100 are likely errors)
df.loc[df['meanpressure'] < 100, 'meanpressure'] = df['meanpressure'].median()

# Scale features
feature_cols = ['meantemp', 'humidity', 'wind_speed', 'meanpressure']
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[feature_cols])
scaled_df = pd.DataFrame(scaled_data, columns=feature_cols)

print(f"Scaled data shape: {scaled_df.shape}")
print(f"Scaled data range: [{scaled_data.min():.4f}, {scaled_data.max():.4f}]")

# Create sequences for time series
def create_sequences(data, target_col_idx, window_size=30):
    """Create input sequences and targets for time series prediction.
    
    Args:
        data: Scaled numpy array of all features
        target_col_idx: Index of target column in data
        window_size: Number of past timesteps to use as input
    
    Returns:
        X: Input sequences (samples, window_size, features)
        y: Target values (samples,)
    """
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i - window_size:i])
        y.append(data[i, target_col_idx])
    return np.array(X), np.array(y)

WINDOW_SIZE = 30
target_idx = feature_cols.index('meantemp')
X, y = create_sequences(scaled_data, target_idx, WINDOW_SIZE)

# Split (keep temporal order - no shuffling for time series!)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# For ANN: flatten the sequences
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print(f"\nSequence shape - X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Flattened - X_train_flat: {X_train_flat.shape}")


---
## 4. 🏗️ Model Building

In [ ]:

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
EPOCHS = 150
BATCH = 32
n_features = X_train.shape[2]
results = {}


### 4.1 ANN

In [ ]:
print("\n" + "=" * 50)
print("MODEL 1: ANN")
print("=" * 50)

model_ann = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_flat.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)
])
model_ann.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_ann.summary()

history_ann = model_ann.fit(X_train_flat, y_train, epochs=EPOCHS, batch_size=BATCH,
                            validation_split=0.2, callbacks=[early_stop], verbose=1)


### 4.2 Simple RNN

In [ ]:
print("\n" + "=" * 50)
print("MODEL 2: Simple RNN")
print("=" * 50)

model_rnn = Sequential([
    SimpleRNN(128, activation='tanh', input_shape=(WINDOW_SIZE, n_features), return_sequences=True),
    Dropout(0.3),
    SimpleRNN(64, activation='tanh'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)
])
model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_rnn.summary()

history_rnn = model_rnn.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH,
                            validation_split=0.2, callbacks=[early_stop], verbose=1)


### 4.3 LSTM

In [ ]:
print("\n" + "=" * 50)
print("MODEL 3: LSTM")
print("=" * 50)

model_lstm = Sequential([
    LSTM(128, activation='tanh', input_shape=(WINDOW_SIZE, n_features), return_sequences=True),
    Dropout(0.3),
    LSTM(64, activation='tanh'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)
])
model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_lstm.summary()

history_lstm = model_lstm.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH,
                              validation_split=0.2, callbacks=[early_stop], verbose=1)


### 4.4 GRU

In [ ]:
print("\n" + "=" * 50)
print("MODEL 4: GRU")
print("=" * 50)

model_gru = Sequential([
    GRU(128, activation='tanh', input_shape=(WINDOW_SIZE, n_features), return_sequences=True),
    Dropout(0.3),
    GRU(64, activation='tanh'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)
])
model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_gru.summary()

history_gru = model_gru.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH,
                            validation_split=0.2, callbacks=[early_stop], verbose=1)


---
## 5. 📈 Training Curves

In [ ]:

models_info = {'ANN': history_ann, 'RNN': history_rnn, 'LSTM': history_lstm, 'GRU': history_gru}

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
for i, (name, hist) in enumerate(models_info.items()):
    axes[0, i].plot(hist.history['mae'], label='Train', linewidth=2)
    axes[0, i].plot(hist.history['val_mae'], label='Validation', linewidth=2)
    axes[0, i].set_title(f'{name} - MAE', fontweight='bold')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)
    
    axes[1, i].plot(hist.history['loss'], label='Train', linewidth=2)
    axes[1, i].plot(hist.history['val_loss'], label='Validation', linewidth=2)
    axes[1, i].set_title(f'{name} - Loss (MSE)', fontweight='bold')
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Training vs Validation Curves', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('climate_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6. 📋 Evaluation

In [ ]:

def inverse_scale_temp(scaled_vals):
    """Inverse transform only the temperature column."""
    dummy = np.zeros((len(scaled_vals), len(feature_cols)))
    dummy[:, target_idx] = scaled_vals.flatten()
    return scaler.inverse_transform(dummy)[:, target_idx]

def evaluate_regression(model, X_te, y_te, model_name, is_flat=False):
    y_pred_scaled = model.predict(X_te).flatten()
    y_pred = inverse_scale_temp(y_pred_scaled)
    y_actual = inverse_scale_temp(y_te)
    
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
    mae = mean_absolute_error(y_actual, y_pred)
    r2 = r2_score(y_actual, y_pred)
    
    print(f"\n{'=' * 40}")
    print(f"  {model_name} Results")
    print(f"{'=' * 40}")
    print(f"  RMSE:  {rmse:.4f}")
    print(f"  MAE:   {mae:.4f}")
    print(f"  R²:    {r2:.4f}")
    
    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2,
            'y_pred': y_pred, 'y_actual': y_actual}

r_ann = evaluate_regression(model_ann, X_test_flat, y_test, 'ANN', True)
r_rnn = evaluate_regression(model_rnn, X_test, y_test, 'RNN')
r_lstm = evaluate_regression(model_lstm, X_test, y_test, 'LSTM')
r_gru = evaluate_regression(model_gru, X_test, y_test, 'GRU')

all_results = [r_ann, r_rnn, r_lstm, r_gru]


---
## 7. 📊 Visualization & Comparison

In [ ]:

# Prediction vs Actual plots
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for i, r in enumerate(all_results):
    ax = axes[i // 2, i % 2]
    ax.plot(r['y_actual'], label='Actual', linewidth=1.5, alpha=0.8)
    ax.plot(r['y_pred'], label='Predicted', linewidth=1.5, alpha=0.8)
    ax.set_title(f"{r['Model']} - Prediction vs Actual", fontweight='bold')
    ax.set_xlabel('Time Steps')
    ax.set_ylabel('Temperature (°C)')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle('Temperature Forecasting Results', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('climate_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

# Comparison table
comp_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['y_pred', 'y_actual']} 
                        for r in all_results]).set_index('Model')
print("\n" + "=" * 60)
print("  MODEL COMPARISON TABLE")
print("=" * 60)
print(comp_df.round(4).to_string())

# Bar chart
comp_df[['RMSE', 'MAE']].plot(kind='bar', figsize=(10, 5), edgecolor='black')
plt.title('Model Error Comparison (Lower is Better)', fontweight='bold')
plt.ylabel('Error')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('climate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best = comp_df['R²'].idxmax()
print(f"\n🏆 Best model: {best} (R² = {comp_df.loc[best, 'R²']:.4f})")

# Comparative Discussion
print("""
COMPARATIVE ANALYSIS:
- LSTM/GRU typically outperform Simple RNN on time series due to gating mechanisms
  that solve the vanishing gradient problem.
- ANN may struggle with temporal dependencies since it treats flattened input as
  independent features without sequential context.
- LSTM has 3 gates (forget, input, output) - most parameters, highest computational cost.
- GRU has 2 gates (reset, update) - fewer parameters, often comparable to LSTM performance.
- For this Delhi climate dataset, seasonal temperature patterns are strong,
  making even simpler models reasonably effective.
""")


---
## 8. 💾 Model Saving

In [ ]:

os.makedirs('saved_models', exist_ok=True)

model_ann.save('saved_models/climate_ann_model.keras')
model_rnn.save('saved_models/climate_rnn_model.keras')
model_lstm.save('saved_models/climate_lstm_model.keras')
model_gru.save('saved_models/climate_gru_model.keras')

with open('saved_models/climate_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('saved_models/climate_config.pkl', 'wb') as f:
    pickle.dump({'window_size': WINDOW_SIZE, 'feature_cols': feature_cols, 'target_idx': target_idx}, f)

comp_df.to_csv('saved_models/climate_comparison_results.csv')

print("✅ All models and artifacts saved to saved_models/")
print("🎉 Daily Climate Time Series Notebook Complete!")
